# Problem

Your task in this exercise is to design an empirical evaluation of the two GNN layers from Problem 3 in Python.

1. **Choose two datasets from TUDataset.**  
   To ensure that your evaluation is representative, select datasets from different domains (for example, bioinformatics and social networks).  
   Unless you have access to a lot of compute, choose datasets with at most 2000 graphs and at most 30 nodes per graph.

2. **Design your evaluation protocol**, in particular a strategy for model selection.  
   Keep training, validation, and test splits separate and choose hyperparameters based on the best validation performance. Repeat the experiments across multiple random initializations (> 2).

3. **Implement the GNNs.** Implement the GNN layers \(f\) and \(g\), which update the feature of a vertex \(v\) at the \((t+1)\)-th layer via

   $$
   f^{(t+1)}(v) := \sigma\left(f^{(t)}(v)\mathbf{W}_1 + \sum_{w \in N(v)} f^{(t)}(w)\mathbf{W}_2\right),
   $$

   and

   $$
   g^{(t+1)}(v) := \sigma\left(g^{(t)}(v)\mathbf{W}_1' + \frac{1}{|N(v)|}\sum_{w \in N(v)} g^{(t)}(w)\mathbf{W}_2'\right),
   $$

   respectively.

   Here, \(\sigma\) is the activation function and the \(W\) matrices are learnable parameters.  
   For graph-level tasks, use a pooling layer after the last GNN layer to aggregate all node features into a single graph feature. Typical choices are **sum** or **mean** pooling.

4. **Run a hyper-parameter sweep**, for example over:
   - activation functions,
   - pooling layers,
   - number of layers,
   - embedding dimension \(d\).

   Report the test performance of each model with its best validation score for both datasets, including the standard deviation across seeds. What do you observe?

5. **Document your experimental setup** sufficiently well and report the experimental results.


## Notebook contents

This notebook provides:

- dataset selection from two different TUDataset domains,
- custom implementations of the two required GNN layers,
- train/validation/test splitting,
- repeated runs across multiple seeds,
- hyperparameter sweep,
- summary tables for best validation and corresponding test accuracy.

### Suggested datasets
This notebook uses:

- **MUTAG** (bioinformatics),
- **IMDB-BINARY** (social networks).

These are common small TUDataset benchmarks and fit the assignment constraints well.


In [ ]:
!pip install torch torch-geometric

In [ ]:
import random
from itertools import product

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_add_pool, global_mean_pool
from torch_geometric.utils import degree


In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cpu


In [ ]:
def ensure_node_features(dataset):
    processed = []
    max_degree_seen = 0

    if dataset.num_features == 0:
        for data in dataset:
            deg = degree(data.edge_index[0], num_nodes=data.num_nodes, dtype=torch.long)
            if deg.numel() > 0:
                max_degree_seen = max(max_degree_seen, int(deg.max().item()))

    for data in dataset:
        data = data.clone()
        if data.x is None:
            deg = degree(data.edge_index[0], num_nodes=data.num_nodes, dtype=torch.long)
            x = F.one_hot(deg.clamp(max=max_degree_seen), num_classes=max_degree_seen + 1).float()
            data.x = x
        processed.append(data)

    num_features = processed[0].x.size(-1)
    return processed, num_features

def load_tu_dataset(name: str):
    dataset = TUDataset(root=f"/tmp/{name}", name=name)
    data_list, num_features = ensure_node_features(dataset)
    num_classes = dataset.num_classes
    return data_list, num_features, num_classes

dataset_names = ["MUTAG", "IMDB-BINARY"]
datasets = {}

for name in dataset_names:
    data_list, num_features, num_classes = load_tu_dataset(name)
    datasets[name] = {
        "data_list": data_list,
        "num_features": num_features,
        "num_classes": num_classes,
        "num_graphs": len(data_list),
        "avg_nodes": float(np.mean([d.num_nodes for d in data_list])),
    }

pd.DataFrame(
    [
        {
            "dataset": name,
            "graphs": info["num_graphs"],
            "num_features": info["num_features"],
            "num_classes": info["num_classes"],
            "avg_nodes": round(info["avg_nodes"], 2),
        }
        for name, info in datasets.items()
    ]
)


Processing...
Done!
Processing...
Done!


,dataset,graphs,num_features,num_classes,avg_nodes
0,MUTAG,188,7,2,17.93
1,IMDB-BINARY,1000,136,2,19.77


In [ ]:
def split_indices(n, seed, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)

    n_train = int(train_ratio * n)
    n_val = int(val_ratio * n)

    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train + n_val]
    test_idx = idx[n_train + n_val:]
    return train_idx, val_idx, test_idx

def make_loaders(data_list, seed, batch_size=64):
    train_idx, val_idx, test_idx = split_indices(len(data_list), seed)
    train_set = [data_list[i] for i in train_idx]
    val_set = [data_list[i] for i in val_idx]
    test_set = [data_list[i] for i in test_idx]

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


In [ ]:
class SumGNNLayer(nn.Module):
    # f^{t+1}(v) = sigma( x_v W1 + sum_{w in N(v)} x_w W2 )
    def __init__(self, in_channels, out_channels, activation="relu"):
        super().__init__()
        self.lin_self = nn.Linear(in_channels, out_channels, bias=False)
        self.lin_neigh = nn.Linear(in_channels, out_channels, bias=False)
        self.activation = activation

    def forward(self, x, edge_index):
        row, col = edge_index
        agg = torch.zeros_like(x)
        agg.index_add_(0, row, x[col])

        out = self.lin_self(x) + self.lin_neigh(agg)
        return self.apply_activation(out)

    def apply_activation(self, x):
        if self.activation == "relu":
            return F.relu(x)
        if self.activation == "tanh":
            return torch.tanh(x)
        if self.activation == "sigmoid":
            return torch.sigmoid(x)
        raise ValueError(f"Unsupported activation: {self.activation}")


class MeanGNNLayer(nn.Module):
    # g^{t+1}(v) = sigma( x_v W1 + (1/|N(v)|) sum_{w in N(v)} x_w W2 )
    def __init__(self, in_channels, out_channels, activation="relu"):
        super().__init__()
        self.lin_self = nn.Linear(in_channels, out_channels, bias=False)
        self.lin_neigh = nn.Linear(in_channels, out_channels, bias=False)
        self.activation = activation

    def forward(self, x, edge_index):
        row, col = edge_index
        agg = torch.zeros_like(x)
        agg.index_add_(0, row, x[col])

        deg = degree(row, num_nodes=x.size(0), dtype=x.dtype).clamp(min=1).unsqueeze(-1)
        agg = agg / deg

        out = self.lin_self(x) + self.lin_neigh(agg)
        return self.apply_activation(out)

    def apply_activation(self, x):
        if self.activation == "relu":
            return F.relu(x)
        if self.activation == "tanh":
            return torch.tanh(x)
        if self.activation == "sigmoid":
            return torch.sigmoid(x)
        raise ValueError(f"Unsupported activation: {self.activation}")


class GraphClassifier(nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_dim,
        num_classes,
        num_layers=2,
        layer_type="sum",
        activation="relu",
        pooling="sum",
    ):
        super().__init__()

        layer_cls = SumGNNLayer if layer_type == "sum" else MeanGNNLayer

        layers = []
        dims = [in_channels] + [hidden_dim] * num_layers
        for i in range(num_layers):
            layers.append(layer_cls(dims[i], dims[i + 1], activation=activation))
        self.layers = nn.ModuleList(layers)

        self.pooling = pooling
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        for layer in self.layers:
            x = layer(x, edge_index)

        if self.pooling == "sum":
            graph_emb = global_add_pool(x, batch)
        elif self.pooling == "mean":
            graph_emb = global_mean_pool(x, batch)
        else:
            raise ValueError(f"Unsupported pooling: {self.pooling}")

        return self.classifier(graph_emb)


In [ ]:
def accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch)
            pred = logits.argmax(dim=-1)
            correct += int((pred == batch.y).sum().item())
            total += batch.y.size(0)

    return correct / total if total > 0 else 0.0


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    total_graphs = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        logits = model(batch)
        loss = F.cross_entropy(logits, batch.y)
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item()) * batch.y.size(0)
        total_graphs += batch.y.size(0)

    return total_loss / total_graphs if total_graphs > 0 else 0.0


def run_single_experiment(
    data_list,
    num_features,
    num_classes,
    seed,
    layer_type="sum",
    activation="relu",
    pooling="sum",
    hidden_dim=64,
    num_layers=2,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=64,
    epochs=50,
):
    set_seed(seed)
    train_loader, val_loader, test_loader = make_loaders(data_list, seed=seed, batch_size=batch_size)

    model = GraphClassifier(
        in_channels=num_features,
        hidden_dim=hidden_dim,
        num_classes=num_classes,
        num_layers=num_layers,
        layer_type=layer_type,
        activation=activation,
        pooling=pooling,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val = -1.0
    best_test = -1.0
    best_epoch = -1

    history = []

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer)
        val_acc = accuracy(model, val_loader)
        test_acc = accuracy(model, test_loader)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_acc": val_acc,
            "test_acc": test_acc,
        })

        if val_acc > best_val:
            best_val = val_acc
            best_test = test_acc
            best_epoch = epoch

    return {
        "best_val_acc": best_val,
        "test_acc_at_best_val": best_test,
        "best_epoch": best_epoch,
        "history": history,
    }


In [ ]:
search_space = {
    "layer_type": ["sum", "mean"],
    "activation": ["relu", "tanh"],
    "pooling": ["sum", "mean"],
    "num_layers": [2, 3],
    "hidden_dim": [32, 64],
}

seeds = [0, 1, 2]
epochs = 50

configs = [
    dict(zip(search_space.keys(), values))
    for values in product(*search_space.values())
]

print("Number of configurations:", len(configs))


Number of configurations: 32


In [ ]:
all_results = []

for dataset_name, info in datasets.items():
    print(f"\nDataset: {dataset_name}")
    data_list = info["data_list"]
    num_features = info["num_features"]
    num_classes = info["num_classes"]

    for config_id, config in enumerate(configs, start=1):
        seed_results = []

        for seed in seeds:
            result = run_single_experiment(
                data_list=data_list,
                num_features=num_features,
                num_classes=num_classes,
                seed=seed,
                layer_type=config["layer_type"],
                activation=config["activation"],
                pooling=config["pooling"],
                hidden_dim=config["hidden_dim"],
                num_layers=config["num_layers"],
                epochs=epochs,
            )
            seed_results.append(result)

        row = {
            "dataset": dataset_name,
            **config,
            "val_mean": float(np.mean([r["best_val_acc"] for r in seed_results])),
            "val_std": float(np.std([r["best_val_acc"] for r in seed_results])),
            "test_mean": float(np.mean([r["test_acc_at_best_val"] for r in seed_results])),
            "test_std": float(np.std([r["test_acc_at_best_val"] for r in seed_results])),
            "best_epoch_mean": float(np.mean([r["best_epoch"] for r in seed_results])),
        }
        all_results.append(row)

results_df = pd.DataFrame(all_results)
results_df.sort_values(["dataset", "val_mean"], ascending=[True, False]).head(10)



Dataset: MUTAG

Dataset: IMDB-BINARY


,dataset,layer_type,activation,pooling,num_layers,hidden_dim,val_mean,val_std,test_mean,test_std,best_epoch_mean
48,IMDB-BINARY,mean,relu,sum,2,32,0.793333,0.028674,0.723333,0.053125,11.000000
51,IMDB-BINARY,mean,relu,sum,3,64,0.793333,0.016997,0.693333,0.020548,11.666667
37,IMDB-BINARY,sum,relu,mean,2,64,0.786667,0.024944,0.710000,0.029439,7.000000
49,IMDB-BINARY,mean,relu,sum,2,64,0.786667,0.012472,0.736667,0.024944,26.666667
47,IMDB-BINARY,sum,tanh,mean,3,64,0.783333,0.016997,0.703333,0.012472,4.000000
50,IMDB-BINARY,mean,relu,sum,3,32,0.780000,0.016330,0.700000,0.008165,7.666667
53,IMDB-BINARY,mean,relu,mean,2,64,0.780000,0.008165,0.716667,0.016997,19.333333
55,IMDB-BINARY,mean,relu,mean,3,64,0.780000,0.016330,0.713333,0.009428,16.333333
36,IMDB-BINARY,sum,relu,mean,2,32,0.773333,0.016997,0.710000,0.037417,8.333333
44,IMDB-BINARY,sum,tanh,mean,2,32,0.773333,0.016997,0.716667,0.023570,10.333333


In [ ]:
best_per_dataset = (
    results_df.sort_values(["dataset", "val_mean"], ascending=[True, False])
    .groupby("dataset")
    .head(1)
    .reset_index(drop=True)
)

best_per_dataset


,dataset,layer_type,activation,pooling,num_layers,hidden_dim,val_mean,val_std,test_mean,test_std,best_epoch_mean
0,IMDB-BINARY,mean,relu,sum,2,32,0.793333,0.028674,0.723333,0.053125,11.0
1,MUTAG,sum,tanh,sum,3,64,0.962963,0.026189,0.700000,0.147196,18.0


## What to report

For each dataset, report:

- the best hyperparameter configuration according to **validation accuracy**,
- the corresponding **test accuracy**,
- the **standard deviation across seeds**.

### Typical observations to discuss

I should expect to comment on questions such as:

- Does the **sum-based layer** or the **mean-based layer** work better on a given dataset?
- Does **sum pooling** or **mean pooling** work better?
- How sensitive are results to the **number of layers** and **hidden dimension**?
- Are the results stable across seeds, or is the standard deviation large?
- Do the two datasets favor different configurations?


In [ ]:
top5 = (
    results_df.sort_values(["dataset", "val_mean"], ascending=[True, False])
    .groupby("dataset")
    .head(5)
    .reset_index(drop=True)
)

top5


,dataset,layer_type,activation,pooling,num_layers,hidden_dim,val_mean,val_std,test_mean,test_std,best_epoch_mean
0,IMDB-BINARY,mean,relu,sum,2,32,0.793333,2.867442e-02,0.723333,0.053125,11.000000
1,IMDB-BINARY,mean,relu,sum,3,64,0.793333,1.699673e-02,0.693333,0.020548,11.666667
2,IMDB-BINARY,sum,relu,mean,2,64,0.786667,2.494438e-02,0.710000,0.029439,7.000000
3,IMDB-BINARY,mean,relu,sum,2,64,0.786667,1.247219e-02,0.736667,0.024944,26.666667
4,IMDB-BINARY,sum,tanh,mean,3,64,0.783333,1.699673e-02,0.703333,0.012472,4.000000
5,MUTAG,sum,tanh,sum,3,64,0.962963,2.618914e-02,0.700000,0.147196,18.000000
6,MUTAG,sum,relu,sum,3,64,0.944444,1.110223e-16,0.650000,0.108012,27.000000
7,MUTAG,sum,tanh,sum,3,32,0.944444,1.110223e-16,0.650000,0.108012,20.666667
8,MUTAG,sum,tanh,mean,3,64,0.944444,1.110223e-16,0.666667,0.084984,33.333333
9,MUTAG,sum,relu,sum,3,32,0.925926,2.618914e-02,0.650000,0.081650,19.666667


## Result-based interpretation

The results show that the two datasets prefer different settings. On IMDB-BINARY, the mean-based layer works better on average, and mean pooling is also better on average. At the same time, the best single configuration uses the mean-based layer with sum pooling. The number of layers and the hidden dimension do not change the performance much on this dataset, so IMDB-BINARY is not very sensitive to these choices. The results are also fairly stable across seeds, since both the validation and test standard deviations are low overall. On MUTAG, the pattern is different. The sum-based layer works better on average, and sum pooling also works better. MUTAG is much more sensitive to the number of layers and the hidden dimension, because the gaps between settings are clearly larger. The results on MUTAG also change more across seeds, so they are less stable. Overall, IMDB-BINARY seems easier to tune and more stable, while MUTAG depends more strongly on the exact configuration.


